# Hybrid Intelligence for Nursing Education
## Demo Walkthrough: Educator-in-the-Loop HITL Simulation

This notebook walks through the full pipeline of the paper step by step.
It is intended for readers who want to understand the methodology interactively.

**For full-scale experiments**, run the scripts in `src/` instead:
```
python src/01_data_preparation.py
python src/02_baselines.py
python src/03_transformers.py
python src/04_hitl.py
```

---

### Pipeline Overview

| Stage | Script | Description |
|-------|--------|-------------|
| 1 | `01_data_preparation.py` | Load MedNLI, map labels, generate BioGPT rationales |
| 2 | `02_baselines.py` | Train ML baselines (LR, RF, XGBoost) with TF-IDF |
| 3 | `03_transformers.py` | Fine-tune transformers without HITL |
| 4 | `04_hitl.py` | Incremental HITL fine-tuning across 20 rounds |

## 0. Setup

Make sure you have installed the dependencies and are running this notebook
from the `HITL_NursingAI/` root directory.

```bash
pip install -r requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, 'src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from config import DATA_DIR, LABEL2ID, ID2LABEL, HITL_ROUNDS
from utils import set_seed, load_split, build_input_text, compute_metrics

set_seed(42)
print('Setup complete.')

---
## 1. Data Preparation

The MedNLI dataset contains ~14,000 clinical inference records.
Each record has a clinical scenario (premise) and a hypothesis (student judgment).

We map the original MedNLI labels to nursing safety categories:

| MedNLI label   | Nursing label | Meaning                        |
|----------------|---------------|--------------------------------|
| entailment     | safe (0)      | Judgment is clinically sound   |
| contradiction  | unsafe (1)    | Judgment is clinically unsafe  |
| neutral        | ambiguous (2) | Judgment requires further review |

Then BioGPT generates a **rationale** for each record in one of three student voices:
- **Novice** (unsafe): simple language, uncertain phrasing
- **Clinical** (ambiguous): structured reasoning, moderate vocabulary
- **Confident** (safe): concise, authoritative clinical justification

The final model input concatenates all three fields:
```
<scenario> [SEP] <judgment> [SEP] <rationale>
```

In [ ]:
# Load the preprocessed training split
train_df = load_split(DATA_DIR / 'train_full_with_rationales.csv')
val_df   = load_split(DATA_DIR / 'validation_full_with_rationales.csv')
test_df  = load_split(DATA_DIR / 'test_full_with_rationales.csv')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
train_df.head(3)

In [ ]:
# Label distribution
label_counts = train_df['label'].map(ID2LABEL).value_counts()
print('Training label distribution:')
print(label_counts.to_string())
print(f'\nClass balance: {label_counts.std():.1f} std — classes are approximately balanced')

In [ ]:
# Example: what a model input looks like
sample = train_df.iloc[0]
print('=== Sample record ===')
print(f'Scenario   : {sample["sentence1"][:120]}...')
print(f'Judgment   : {sample["sentence2"]}')
print(f'Rationale  : {str(sample.get("rationale", ""))[:120]}...')
print(f'Label      : {ID2LABEL[sample["label"]]} ({sample["label"]})')
print(f'Voice      : {sample.get("voice", "N/A")}')
print()
print('=== Model input (concatenated) ===')
print(build_input_text(sample)[:300], '...')

---
## 2. Baseline ML Models

Three classical ML models are trained with TF-IDF features as a lower bound.
These establish that classical models achieve moderate performance but lack
the semantic depth needed for clinical reasoning classification.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Build inputs
train_texts = [build_input_text(r) for _, r in train_df.iterrows()]
test_texts  = [build_input_text(r) for _, r in test_df.iterrows()]
y_train = train_df['label'].tolist()
y_test  = test_df['label'].tolist()

# TF-IDF
vec     = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train_texts)
X_test  = vec.transform(test_texts)

print(f'TF-IDF vocabulary size: {len(vec.vocabulary_):,}')

In [ ]:
baseline_results = {}

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, random_state=42,
                                         use_label_encoder=False, eval_metric='mlogloss'),
}

for name, clf in models.items():
    print(f'Training {name}...')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)
    m = compute_metrics(np.array(y_test), np.array(y_pred), np.array(y_prob))
    baseline_results[name] = m
    print(f'  Accuracy: {m["accuracy"]:.3f} | F1: {m["macro_f1"]:.3f} | AUC: {m["auc"]:.3f}')

print('\nBaseline evaluation complete.')

In [ ]:
# Summary table
baseline_table = pd.DataFrame(baseline_results).T
baseline_table.index.name = 'Model'
baseline_table.columns = ['Accuracy', 'Macro F1', 'AUC']
baseline_table.round(4)

---
## 3. Transformer Baselines (No HITL)

Five transformer models are fine-tuned **without any HITL** to establish
pre-HITL performance. The full fine-tuning run takes ~2 hours on a GPU.

For this demo we illustrate the setup for **one model** (BioBERT).
To run all five, use `python src/03_transformers.py`.

The key design choices:
- Input: concatenated scenario + judgment + rationale (max 256 tokens)
- `AutoModelForSequenceClassification` works for both BERT-family and T5
- Best checkpoint selected by macro F1 on the validation set

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from utils import ClinicalDataset, make_hf_compute_metrics
from config import NUM_LABELS, ID2LABEL, LABEL2ID, MAX_INPUT_LENGTH

MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.1'
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
def make_dataset(df, tokenizer):
    texts = [build_input_text(r) for _, r in df.iterrows()]
    enc   = tokenizer(texts, padding=True, truncation=True,
                      max_length=MAX_INPUT_LENGTH, return_tensors='pt')
    return ClinicalDataset(enc, df['label'].tolist())

train_dataset = make_dataset(train_df, tokenizer)
val_dataset   = make_dataset(val_df,   tokenizer)
test_dataset  = make_dataset(test_df,  tokenizer)

print(f'Datasets ready — train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}')

In [ ]:
training_args = TrainingArguments(
    output_dir                   = 'models/biobert_demo',
    num_train_epochs             = 3,
    per_device_train_batch_size  = 16,
    per_device_eval_batch_size   = 16,
    learning_rate                = 2e-5,
    warmup_ratio                 = 0.1,
    weight_decay                 = 0.01,
    evaluation_strategy          = 'epoch',
    save_strategy                = 'best',
    load_best_model_at_end       = True,
    metric_for_best_model        = 'macro_f1',
    fp16                         = torch.cuda.is_available(),
    seed                         = 42,
    logging_steps                = 50,
    report_to                    = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = make_hf_compute_metrics(),
)

print('Training configuration ready.')
print('Run trainer.train() to fine-tune — takes ~30 min on GPU for BioBERT.')
# trainer.train()   # ← uncomment to run

### Expected results (from paper)

| Model        | Accuracy | Macro F1 | AUC   |
|--------------|----------|----------|-------|
| T5-small     | 0.910    | 0.91     | 0.929 |
| ClinicalBERT | 0.890    | 0.89     | 0.977 |
| BioBERT      | 0.870    | 0.87     | 0.968 |
| RoBERTa      | 0.860    | 0.86     | 0.962 |
| DistilBERT   | 0.860    | 0.86     | 0.980 |

All transformer models substantially outperform the classical baselines,
confirming they are the right candidates for HITL refinement.

---
## 4. HITL Incremental Fine-Tuning

This is the core contribution of the paper. The HITL workflow has two zones:

**Zone 1 — One-time baseline (runs once):**
1. Split training set → seed (70%) and pool (30%)
2. Fine-tune model on seed set → θ₀

**Zone 2 — Repeating loop (runs R times):**
3. Predict labels for the pool set
4. Identify 150 misclassified examples (simulated educator corrections)
5. Relabel them with ground-truth (the 'correction')
6. Incrementally fine-tune on corrected samples only:
   `θ_{t+1} = θ_t − η ∇L(D_corr)`
7. Evaluate on the held-out test set
8. Record metrics → learning curve

The test set is **never updated** during any HITL round.

In [ ]:
# Demonstrate the seed / pool split
seed_df = train_df.sample(frac=0.70, random_state=42).reset_index(drop=True)
pool_df = train_df.drop(seed_df.index).reset_index(drop=True)

print(f'Seed set : {len(seed_df):,} samples  (used for baseline training θ₀)')
print(f'Pool set : {len(pool_df):,} samples  (corrections drawn from here each round)')
print(f'Test set : {len(test_df):,} samples  (held-out, never touched during HITL)')

In [ ]:
# Demonstrate the correction selection logic (no model needed)
# In practice, y_pred comes from the model — here we simulate it

np.random.seed(42)
y_true_demo = pool_df['label'].to_numpy()[:500]

# Simulate a model with ~85% accuracy
y_pred_demo = y_true_demo.copy()
error_idx   = np.random.choice(len(y_pred_demo), size=75, replace=False)
y_pred_demo[error_idx] = (y_pred_demo[error_idx] + 1) % 3

wrong_mask   = (y_pred_demo != y_true_demo)
n_errors     = wrong_mask.sum()
n_correction = min(150, n_errors)

print(f'Pool sample size      : 500')
print(f'Misclassified         : {n_errors}')
print(f'Selected for correction: {n_correction}')
print(f'Pool remaining        : {500 - n_correction}')
print()
print('The corrections are relabelled with ground-truth labels —')
print('this simulates what an educator would do when reviewing errors.')

In [ ]:
# Visualise the learning curves from the paper results
# (replace with actual results from results/hitl_results.json after running 04_hitl.py)

import json
from pathlib import Path

results_path = Path('results/hitl_results.json')

if results_path.exists():
    with open(results_path) as f:
        hitl_results = json.load(f)
    print('Loaded actual HITL results.')
else:
    # Representative curves from the paper for illustration
    print('results/hitl_results.json not found — using representative curves from paper.')
    rounds = list(range(21))
    hitl_results = {
        't5-small': {
            'round':    rounds,
            'accuracy': [0.51 + 0.019*r - 0.0003*r**2 for r in rounds],
        },
        'biobert': {
            'round':    rounds,
            'accuracy': [0.82 + 0.0025*r - 0.00005*r**2 for r in rounds],
        },
        'clinicalbert': {
            'round':    rounds,
            'accuracy': [0.79 + 0.0035*r - 0.00006*r**2 for r in rounds],
        },
    }

fig, ax = plt.subplots(figsize=(9, 5))
colors  = {'t5-small': 'steelblue', 'biobert': 'darkorange', 'clinicalbert': 'forestgreen'}

for model_key, curves in hitl_results.items():
    rounds   = curves['round']
    accuracy = curves['accuracy']
    ax.plot(rounds, accuracy, marker='o', markersize=4,
            label=model_key, color=colors.get(model_key, 'gray'))
    ax.axhline(accuracy[0], linestyle='--', alpha=0.4,
               color=colors.get(model_key, 'gray'))

ax.set_xlabel('HITL Round', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('HITL Learning Curves — All Models', fontsize=13)
ax.set_xticks(range(0, 21, 2))
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
plt.savefig('results/figures/learning_curves_demo.png', dpi=150)
plt.show()
print('Learning curves saved to results/figures/learning_curves_demo.png')

---
## 5. Key Findings

The dashed lines above show the pre-HITL baseline for each model.
After 20 rounds of incremental correction:

- All three transformer models surpass their pre-HITL baselines
- Improvements are largest in the early rounds (correcting systematic errors)
- Later rounds produce smaller but stable gains
- The optimised incremental strategy (fine-tuning on corrections only,
  not the full accumulated dataset) keeps computation constant across rounds

### HITL vs data augmentation

The key distinction is the **selection logic**. Data augmentation adds more
examples to improve coverage. HITL selects *the model's own errors* and
injects the educator's correction — targeting systematic reasoning failures
rather than simply expanding the training distribution.

In [ ]:
# Final summary table
summary = {
    'Logistic Regression': {'Accuracy': 0.712, 'Macro F1': 0.71,  'AUC': 0.868, 'HITL': 'No'},
    'Random Forest':       {'Accuracy': 0.660, 'Macro F1': 0.66,  'AUC': 0.830, 'HITL': 'No'},
    'XGBoost':             {'Accuracy': 0.710, 'Macro F1': 0.71,  'AUC': 0.875, 'HITL': 'No'},
    'T5-small (no HITL)':  {'Accuracy': 0.910, 'Macro F1': 0.91,  'AUC': 0.929, 'HITL': 'No'},
    'BioBERT (no HITL)':   {'Accuracy': 0.870, 'Macro F1': 0.87,  'AUC': 0.968, 'HITL': 'No'},
    'ClinicalBERT (no HITL)': {'Accuracy': 0.890, 'Macro F1': 0.89, 'AUC': 0.977, 'HITL': 'No'},
    'T5-small + HITL':     {'Accuracy': 0.850, 'Macro F1': 0.85,  'AUC': 0.950, 'HITL': 'Yes'},
    'BioBERT + HITL':      {'Accuracy': 0.870, 'Macro F1': 0.87,  'AUC': 0.975, 'HITL': 'Yes'},
    'ClinicalBERT + HITL': {'Accuracy': 0.865, 'Macro F1': 0.865, 'AUC': 0.982, 'HITL': 'Yes'},
}

summary_df = pd.DataFrame(summary).T
summary_df.index.name = 'Model'
print('Note: HITL values are representative — replace with your actual results from results/hitl_results.json')
summary_df

---
## 6. Next Steps

To run the full experiments:

```bash
python src/01_data_preparation.py   # ~2 hrs (BioGPT rationale generation)
python src/02_baselines.py          # ~5 min
python src/03_transformers.py       # ~2 hrs (GPU)
python src/04_hitl.py               # ~3–4 hrs (GPU)
```

To publish trained models:

```bash
python src/upload_to_hf.py --username your_hf_username
```

See `PUBLISHING_GUIDE.md` for step-by-step GitHub and HuggingFace instructions.